In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
def parse_index_array(s):
    """Parse comma-separated indices from string"""
    if pd.isna(s) or s.strip() == '[]':
        return []
    s = s.strip('[]')
    if not s:
        return []
    indices = [int(x.strip()) for x in s.split(',') if x.strip()]
    return indices

# Read manual annotations with custom parsing
manual_data = []
with open('problematic_images.csv', 'r') as f:
    lines = f.readlines()
    
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        
        parts = []
        current = ""
        bracket_depth = 0
        
        for char in line:
            if char == '[':
                bracket_depth += 1
                current += char
            elif char == ']':
                bracket_depth -= 1
                current += char
            elif char == ',' and bracket_depth == 0:
                parts.append(current)
                current = ""
            else:
                current += char
        
        if current:
            parts.append(current)
        
        if len(parts) >= 5:
            manual_data.append({
                'phrase': parts[0],
                'tensor_file': parts[1],
                'type': int(parts[2]),
                'ambiguous_array': parts[3],
                'incorrect_array': parts[4]
            })

manual_df = pd.DataFrame(manual_data)
print(f"Loaded {len(manual_df)} manual annotation rows")
print(f"Phrases: {', '.join(manual_df['phrase'].unique())}")

In [ ]:
# Load all Qwen predictions
qwen_folders = {
    'simple': 'qwen_outputs/simple',
    'reason': 'qwen_outputs/reason',
}

all_qwen_data = []

for folder_name, folder_path in qwen_folders.items():
    csv_files = glob.glob(f'{folder_path}/*.csv')
    
    for csv_file in csv_files:
        filename = Path(csv_file).name
        phrase = filename.split('_qwen_')[0].replace('_', ' ')
        
        df = pd.read_csv(csv_file)
        df['phrase'] = phrase
        df['qwen_folder'] = folder_name
        df['source_file'] = filename
        all_qwen_data.append(df)

qwen_df = pd.concat(all_qwen_data, ignore_index=True)
print(f"Loaded {len(qwen_df)} Qwen predictions")
# print(f"From 'simple' folder: {len(qwen_df[qwen_df['qwen_folder']=='simple'])}")
# print(f"From 'reason' folder: {len(qwen_df[qwen_df['qwen_folder']=='reason'])}")

## 2. Query Function - Lookup by Phrase

In [ ]:
def analyze_phrase(phrase_name):
    """
    Analyze a specific phrase showing:
    - Manual annotations by SD type
    - Qwen predictions (only compared with Type 0)
    """
    print(f"PHRASE: {phrase_name}")
    
    # Get manual data for this phrase
    manual_rows = manual_df[manual_df['phrase'] == phrase_name]
    
    # Get Qwen data for this phrase
    qwen_rows = qwen_df[qwen_df['phrase'] == phrase_name]
    
    if len(manual_rows) == 0:
        # No manual data - show only Qwen predictions
        if len(qwen_rows) == 0:
            print(f"[!] No manual annotations or Qwen predictions found for phrase '{phrase_name}'")
            return
        
        print(f"\nNo manual annotations available. Showing Qwen predictions only.\n")
        print("QWEN PREDICTIONS\n")
        
        # Group by folder and show stats
        for folder in ['simple', 'reason']:
            folder_data = qwen_rows[qwen_rows['qwen_folder'] == folder]
            
            if len(folder_data) == 0:
                continue
            
            print(f"  FOLDER: {folder.upper()}")
            
            # Count by answer type
            answer_counts = folder_data['answer'].value_counts().to_dict()
            print(f"   Answer distribution:")
            for answer, count in sorted(answer_counts.items()):
                print(f"     {answer}: {count} images")
            
            # Show problematic (NO/UNCLEAR/AMBIGUOUS) vs correct (YES)
            problematic = folder_data[folder_data['answer'].isin(['NO', 'UNCLEAR', 'AMBIGUOUS'])]
            correct = folder_data[folder_data['answer'] == 'YES']
            
            print(f"   Categorized:")
            print(f"     Problematic (NO/UNCLEAR/AMBIGUOUS): {len(problematic)} images")
            print(f"     Correct (YES): {len(correct)} images")
        
        return
    
    print("\nMANUAL ANNOTATIONS")
    
    # Only show Type 0 (Normal SD)
    type0_row = manual_rows[manual_rows['type'] == 0]
    
    if len(type0_row) == 0:
        print("No Type 0 (Normal SD) data found")
        return
    
    type0_row = type0_row.iloc[0]
    ambiguous_indices = parse_index_array(type0_row['ambiguous_array'])
    incorrect_indices = parse_index_array(type0_row['incorrect_array'])
    
    print("Normal SD (type=0)")
    print(f"  Ambiguous images: {len(ambiguous_indices)} images")
    print(f"  Incorrect images: {len(incorrect_indices)} images")
    print(f"  Total problematic: {len(ambiguous_indices) + len(incorrect_indices)} images")
    
    if len(qwen_rows) == 0:
        print(f"[!] No Qwen predictions found for phrase '{phrase_name}'")
        return
    
    print("\nQWEN PREDICTIONS\n")
    # Group by folder and show stats
    for folder in ['simple', 'reason']:
        folder_data = qwen_rows[qwen_rows['qwen_folder'] == folder]
        
        if len(folder_data) == 0:
            continue
        
        print(f"  FOLDER: {folder.upper()}")
        
        # Count by answer type
        answer_counts = folder_data['answer'].value_counts().to_dict()
        print(f"   Answer distribution:")
        for answer, count in sorted(answer_counts.items()):
            print(f"     {answer}: {count} images")
        
        # Show problematic (NO/UNCLEAR/AMBIGUOUS) vs correct (YES)
        problematic = folder_data[folder_data['answer'].isin(['NO', 'UNCLEAR', 'AMBIGUOUS'])]
        correct = folder_data[folder_data['answer'] == 'YES']
        
        print(f"   Categorized:")
        print(f"     Problematic (NO/UNCLEAR/AMBIGUOUS): {len(problematic)} images")
        print(f"     Correct (YES): {len(correct)} images")
    
    # Compare with manual for Type 0 only
    print("\nCOMPARISON")
    
    type0_rows = manual_rows[manual_rows['type'] == 0]
    
    if len(type0_rows) == 0:
        print("No Type 0 (Normal SD) data found for this phrase")
        return
    
    for _, manual_row in type0_rows.iterrows():
        tensor_file = manual_row['tensor_file']
        
        ambiguous_indices = set(parse_index_array(manual_row['ambiguous_array']))
        incorrect_indices = set(parse_index_array(manual_row['incorrect_array']))
        manual_problematic_indices = ambiguous_indices | incorrect_indices
        
        # Find Qwen predictions for this tensor file
        qwen_for_tensor = qwen_rows[qwen_rows['tensor_file'] == tensor_file]
        
        if len(qwen_for_tensor) == 0:
            print(f"Normal SD (type=0)")
            print(f"  No Qwen predictions found for this tensor file")
            continue
        
        print(f"Normal SD (type=0)")
        print(f"  Manual: {len(manual_problematic_indices)} problematic images\n")
        
        for folder in ['simple', 'reason']:
            folder_tensor_data = qwen_for_tensor[qwen_for_tensor['qwen_folder'] == folder]
            
            if len(folder_tensor_data) == 0:
                continue
            
            qwen_problematic = folder_tensor_data[
                folder_tensor_data['answer'].isin(['NO', 'UNCLEAR', 'AMBIGUOUS'])
            ]
            qwen_problematic_indices = set(qwen_problematic['tensor_index'].astype(int).tolist())
            
            # Calculate overlap
            true_positive = len(manual_problematic_indices & qwen_problematic_indices)
            false_positive = len(qwen_problematic_indices - manual_problematic_indices)
            false_negative = len(manual_problematic_indices - qwen_problematic_indices)
            
            print(f"Qwen ({folder}):")
            print(f"    Predicted problematic: {len(qwen_problematic_indices)} images")
            print(f"    True Positives: {true_positive} images")
            print(f"    False Positives: {false_positive} images")
            print(f"    False Negatives: {false_negative} images\n")


In [ ]:
# def compare_type0_vs_type1(phrase_name):
#     """
#     Compare manual annotations between Type 0 (Normal SD) and Type 1 (Custom SD)
#     Shows differences in problematic images between the two SD types
#     """
#     print(f"MANUAL COMPARISON: Type 0 vs Type 1 for '{phrase_name}'")
    
#     # Get manual data for this phrase
#     manual_rows = manual_df[manual_df['phrase'] == phrase_name]
    
#     if len(manual_rows) == 0:
#         print(f"No manual data found for phrase '{phrase_name}'")
#         return
    
#     type0_row = manual_rows[manual_rows['type'] == 0]
#     type1_row = manual_rows[manual_rows['type'] == 1]
    
#     if len(type0_row) == 0:
#         print(f"No Type 0 (Normal SD) data found")
#         return
    
#     if len(type1_row) == 0:
#         print(f"No Type 1 (Custom SD) data found")
#         return
    
#     type0_row = type0_row.iloc[0]
#     type1_row = type1_row.iloc[0]
    
#     # Parse indices for Type 0
#     type0_ambiguous = set(parse_index_array(type0_row['ambiguous_array']))
#     type0_incorrect = set(parse_index_array(type0_row['incorrect_array']))
#     type0_problematic = type0_ambiguous | type0_incorrect
    
#     # Parse indices for Type 1
#     type1_ambiguous = set(parse_index_array(type1_row['ambiguous_array']))
#     type1_incorrect = set(parse_index_array(type1_row['incorrect_array']))
#     type1_problematic = type1_ambiguous | type1_incorrect
    
#     # Assume 100 total images (adjust if you know the actual total)
#     total_images = 100
    
#     print(f"\nType 0 (Normal SD) - {type0_row['tensor_file']}")
#     print(f"  Ambiguous: {len(type0_ambiguous)} images, {len(type0_ambiguous)/total_images*100:.1f}% Ambiguous")
#     print(f"  Incorrect: {len(type0_incorrect)} images, {len(type0_incorrect)/total_images*100:.1f}% Incorrect")
#     print(f"  Total problematic: {len(type0_problematic)} images, {len(type0_problematic)/total_images*100:.1f}% problematic\n")
    
#     print(f"Type 1 (Custom SD) - {type1_row['tensor_file']}")
#     print(f"  Ambiguous: {len(type1_ambiguous)} images, {len(type1_ambiguous)/total_images*100:.1f}% Ambiguous")
#     print(f"  Incorrect: {len(type1_incorrect)} images, {len(type1_incorrect)/total_images*100:.1f}% Incorrect")
#     print(f"  Total problematic: {len(type1_problematic)} images, {len(type1_problematic)/total_images*100:.1f}% problematic\n")
    
#     # Compare the two
#     both_problematic = type0_problematic & type1_problematic
#     only_type0 = type0_problematic - type1_problematic
#     only_type1 = type1_problematic - type0_problematic
    
#     print(f"COMPARISON:")
#     print(f"  Problematic in BOTH types: {len(both_problematic)} images")
#     print(f"  Problematic ONLY in Type 0 (Normal SD): {len(only_type0)} images")
#     print(f"  Problematic ONLY in Type 1 (Custom SD): {len(only_type1)} images\n")
    
#     if len(both_problematic) > 0:
#         print(f"  Indices problematic in BOTH:")
#         print(f"    {sorted(both_problematic)}\n")
    
#     if len(only_type0) > 0:
#         print(f"  Indices problematic ONLY in Type 0:")
#         print(f"    {sorted(only_type0)}\n")
    
#     if len(only_type1) > 0:
#         print(f"  Indices problematic ONLY in Type 1:")
#         print(f"    {sorted(only_type1)}\n")
    
#     # Calculate improvement
#     if len(type0_problematic) > 0:
#         improvement_pct = (len(only_type0) - len(only_type1)) / len(type0_problematic) * 100
#         if improvement_pct > 0:
#             print(f"Type 1 (Custom SD) shows {improvement_pct:.1f}% improvement (fewer unique problems)")
#         elif improvement_pct < 0:
#             print(f"Type 1 (Custom SD) shows {abs(improvement_pct):.1f}% degradation (more unique problems)")
#         else:
#             print(f"No net improvement between types")


## 3. Usage Examples

In [ ]:
# List all available phrases
print("Available phrases for query:")
for phrase in sorted(manual_df['phrase'].unique()):
    print(f"  • {phrase}")

In [ ]:
analyze_phrase('above')

In [ ]:
analyze_phrase('below')

In [ ]:
analyze_phrase('to the left of')

In [ ]:
analyze_phrase('to the right of')

In [ ]:
analyze_phrase('on top of')

In [ ]:
analyze_phrase('under')

## 4. Detailed Image-Level Comparison

In [ ]:
def confusion_matrix_analysis(phrase_name, tensor_file=None):
    """
    Generate confusion matrix for manual annotations vs Qwen predictions
    Shows TP, TN, FP, FN for each Qwen folder (simple and reason)
    """
    
    
    # Get manual data for this phrase
    manual_rows = manual_df[manual_df['phrase'] == phrase_name]
    
    if len(manual_rows) == 0:
        print(f"No manual annotations found for phrase '{phrase_name}'")
        return
    
    # Get Qwen data for this phrase
    qwen_rows = qwen_df[qwen_df['phrase'] == phrase_name]
    
    if len(qwen_rows) == 0:
        print(f"No Qwen predictions found for phrase '{phrase_name}'")
        return
    
    # If tensor_file not specified, use all
    if tensor_file is None:
        tensor_files = manual_rows['tensor_file'].unique()
    else:
        tensor_files = [tensor_file]
    
    print(f"CONFUSION MATRIX ANALYSIS: '{phrase_name}'\n")
    
    for tf in tensor_files:
        manual_row = manual_rows[manual_rows['tensor_file'] == tf]
        qwen_data = qwen_rows[qwen_rows['tensor_file'] == tf]
        
        if len(manual_row) == 0:
            continue
        
        manual_row = manual_row.iloc[0]
        sd_type = manual_row['type']
        sd_type_label = "Normal SD (type=0)" if sd_type == 0 else "Custom SD (type=1)"
        
        print(f"Tensor: {tf} ({sd_type_label})")
        
        # Parse manual annotations
        ambiguous = set(parse_index_array(manual_row['ambiguous_array']))
        incorrect = set(parse_index_array(manual_row['incorrect_array']))
        manual_problematic = ambiguous | incorrect
        
        # Get all unique indices
        all_indices = set(qwen_data['tensor_index'].astype(int).tolist())
        
        # Organize Qwen predictions by index
        qwen_by_index = {}
        for _, row in qwen_data.iterrows():
            idx = int(row['tensor_index'])
            folder = row['qwen_folder']
            if idx not in qwen_by_index:
                qwen_by_index[idx] = {}
            qwen_by_index[idx][folder] = row['answer']
        
        # Calculate confusion matrix for each folder
        available_folders = qwen_data['qwen_folder'].unique().tolist()
        
        for folder in ['simple', 'reason']:
            if folder not in available_folders:
                continue
            
            # Initialize confusion matrix
            tp = 0  # Manual=problematic, Qwen=problematic
            tn = 0  # Manual=OK, Qwen=OK
            fp = 0  # Manual=OK, Qwen=problematic
            fn = 0  # Manual=problematic, Qwen=OK
            
            for idx in all_indices:
                manual_prob = idx in manual_problematic
                pred = qwen_by_index.get(idx, {}).get(folder, 'N/A')
                pred_prob = pred in ['NO', 'UNCLEAR', 'AMBIGUOUS']
                
                if manual_prob and pred_prob:
                    tp += 1
                elif not manual_prob and not pred_prob:
                    tn += 1
                elif not manual_prob and pred_prob:
                    fp += 1
                elif manual_prob and not pred_prob:
                    fn += 1
            
            # Calculate metrics
            total = tp + tn + fp + fn
            accuracy = (tp + tn) / total if total > 0 else 0
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            
            # Create confusion matrix array
            cm = np.array([[tn, fp], [fn, tp]])
            
            # Print results
            print(f"\n  Qwen ({folder}):")
            print(f"    Confusion Matrix:")
            print(f"                    Predicted: OK  Predicted: Problematic")
            print(f"    Manual: OK           {tn:<6}     {fp:<6}")
            print(f"    Manual: Problematic  {fn:<6}     {tp:<6}")
            print(f"\n    Metrics:")
            print(f"      Accuracy:    {accuracy:.3f} ({tp+tn}/{total})")
            print(f"      Precision:   {precision:.3f} (TP / (TP+FP) = {tp}/{tp+fp})")
            print(f"      Recall:      {recall:.3f} (TP / (TP+FN) = {tp}/{tp+fn})")
            print(f"      Specificity: {specificity:.3f} (TN / (TN+FP) = {tn}/{tn+fp})")
            print(f"      F1-Score:    {f1:.3f}")
            
            # Visualize
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                       xticklabels=['Predicted: OK', 'Predicted: Problematic'],
                       yticklabels=['Manual: OK', 'Manual: Problematic'],
                       cbar_kws={'label': 'Count'})
            plt.title(f"Confusion Matrix: {phrase_name} - {folder}\n{tf} ({sd_type_label})")
            plt.ylabel('Manual Annotation')
            plt.xlabel('Qwen Prediction')
            plt.tight_layout()
            plt.show()
        
        print("\n" + "="*80 + "\n")


In [ ]:
confusion_matrix_analysis('to the right of')

In [ ]:
confusion_matrix_analysis('to the left of')

In [ ]:
confusion_matrix_analysis('below')

In [ ]:
confusion_matrix_analysis('above')

## 4a. Confusion Matrix Analysis

In [ ]:
def compare_image_level(phrase_name, tensor_file=None):
    """
    Show image-by-image comparison for a specific phrase
    If tensor_file is not specified, it will use all tensor files for that phrase
    """
    # Get manual data for this phrase
    manual_rows = manual_df[manual_df['phrase'] == phrase_name]
    
    # Get Qwen data for this phrase
    qwen_rows = qwen_df[qwen_df['phrase'] == phrase_name]
    
    if len(manual_rows) == 0:
        # No manual data - handle Qwen-only comparison
        if len(qwen_rows) == 0:
            print(f"No manual data or Qwen predictions found for phrase '{phrase_name}'")
            return
        
        # If tensor_file not specified, use all available tensor files from Qwen data
        if tensor_file is None:
            tensor_files = qwen_rows['tensor_file'].unique()
            if len(tensor_files) > 1:
                print(f"Found {len(tensor_files)} tensor files for phrase '{phrase_name}':")
                for tf in tensor_files:
                    print(f"  - {tf}")
                print("\nShowing comparison for all tensor files...")
            
            for tf in tensor_files:
                _compare_single_tensor(phrase_name, tf, None)
                print("")
            return
        else:
            _compare_single_tensor(phrase_name, tensor_file, None)
            return
    
    # If tensor_file not specified and there are multiple, show all
    if tensor_file is None:
        if len(manual_rows) > 1:
            print(f"Found {len(manual_rows)} tensor files for phrase '{phrase_name}':")
            for _, row in manual_rows.iterrows():
                sd_type_name = "Normal SD (type=0)" if row['type'] == 0 else "Custom SD (type=1)"
                print(f"  - {row['tensor_file']} ({sd_type_name})")
            print("\nShowing comparison for all tensor files...")
        
        # Process each tensor file
        for _, manual_row in manual_rows.iterrows():
            _compare_single_tensor(phrase_name, manual_row['tensor_file'], manual_row['type'])
            print("")
        return
    
    # Single tensor file specified - need to find the type
    matching_row = manual_rows[manual_rows['tensor_file'] == tensor_file]
    if len(matching_row) > 0:
        sd_type = matching_row.iloc[0]['type']
    else:
        sd_type = None
    _compare_single_tensor(phrase_name, tensor_file, sd_type)


def _compare_single_tensor(phrase_name, tensor_file, sd_type=None):
    """
    Internal function to compare a single tensor file
    """
    sd_type_label = ""
    if sd_type == 0:
        sd_type_label = " (Normal SD - type=0)"
    elif sd_type == 1:
        sd_type_label = " (Custom SD - type=1)"
    
    print(f"IMAGE-LEVEL COMPARISON: '{phrase_name}' - {tensor_file}{sd_type_label}")
    
    # Get manual data
    manual_row = manual_df[
        (manual_df['phrase'] == phrase_name) & 
        (manual_df['tensor_file'] == tensor_file)
    ]
    
    # Get Qwen data
    qwen_data = qwen_df[
        (qwen_df['phrase'] == phrase_name) & 
        (qwen_df['tensor_file'] == tensor_file)
    ]
    
    if len(qwen_data) == 0:
        print(f"No Qwen predictions found")
        return
    
    # Check which Qwen folders have data for this phrase
    available_folders = qwen_data['qwen_folder'].unique().tolist()
    
    # Get all unique indices
    all_indices = set(qwen_data['tensor_index'].astype(int).tolist())
    
    # Organize Qwen predictions by index
    qwen_by_index = {}
    for _, row in qwen_data.iterrows():
        idx = int(row['tensor_index'])
        folder = row['qwen_folder']
        if idx not in qwen_by_index:
            qwen_by_index[idx] = {}
        qwen_by_index[idx][folder] = row['answer']
    
    if len(manual_row) == 0:
        # No manual data - compare only Qwen simple vs reason
        print(f"\nNo manual annotations available. Comparing Qwen predictions only.\n")
        
        # Calculate agreement between simple and reason
        if 'simple' in available_folders and 'reason' in available_folders:
            agree = 0
            disagree = 0
            
            for idx in all_indices:
                simple_pred = qwen_by_index.get(idx, {}).get('simple', 'N/A')
                reason_pred = qwen_by_index.get(idx, {}).get('reason', 'N/A')
                
                simple_prob = simple_pred in ['NO', 'UNCLEAR', 'AMBIGUOUS']
                reason_prob = reason_pred in ['NO', 'UNCLEAR', 'AMBIGUOUS']
                
                if simple_prob == reason_prob:
                    agree += 1
                else:
                    disagree += 1
            
            total = len(all_indices)
            print(f"COMPARISON BETWEEN QWEN FOLDERS (Total images: {total})")
            print(f"  Agreement: {agree} images ({100*agree/total:.1f}%)")
            print(f"  Disagreement: {disagree} images ({100*disagree/total:.1f}%)")
            
            print(f"\nDETAILED COMPARISON")
            print(f"{'Idx':<5} {'Qwen(simple)':<15} {'Qwen(reason)':<15} {'Status':<20}")
            
            for idx in sorted(all_indices):
                simple_pred = qwen_by_index.get(idx, {}).get('simple', 'N/A')
                reason_pred = qwen_by_index.get(idx, {}).get('reason', 'N/A')
                
                simple_prob = simple_pred in ['NO', 'UNCLEAR', 'AMBIGUOUS']
                reason_prob = reason_pred in ['NO', 'UNCLEAR', 'AMBIGUOUS']
                
                if simple_prob == reason_prob:
                    if simple_prob:
                        status = "Agree (both prob)"
                    else:
                        status = "Agree (both OK)"
                else:
                    status = "Disagree"
                
                print(f"{idx:<5} {simple_pred:<15} {reason_pred:<15} {status:<20}")
        else:
            # Only one folder available
            print(f"Only {available_folders[0]} predictions available. No comparison possible.")
        
        return
    
    manual_row = manual_row.iloc[0]
    ambiguous = set(parse_index_array(manual_row['ambiguous_array']))
    incorrect = set(parse_index_array(manual_row['incorrect_array']))
    
    # Calculate statistics with precision and recall for available folders
    stats = {}
    for folder in ['simple', 'reason']:
        if folder not in available_folders:
            continue
            
        stats[folder] = {
            'match': 0,
            'differ': 0,
            'tp': 0,
            'fp': 0,
            'fn': 0
        }
    
    for idx in all_indices:
        # Manual label
        if idx in incorrect or idx in ambiguous:
            manual_prob = True
        else:
            manual_prob = False
        
        # Check each available folder
        for folder in available_folders:
            pred = qwen_by_index.get(idx, {}).get(folder, 'N/A')
            pred_prob = pred in ['NO', 'UNCLEAR', 'AMBIGUOUS']
            
            # Count matches and confusion matrix values
            if manual_prob == pred_prob:
                stats[folder]['match'] += 1
            else:
                stats[folder]['differ'] += 1
            
            if manual_prob and pred_prob:
                stats[folder]['tp'] += 1
            elif not manual_prob and pred_prob:
                stats[folder]['fp'] += 1
            elif manual_prob and not pred_prob:
                stats[folder]['fn'] += 1
    
    total = len(all_indices)
    
    # Print statistics header
    print(f"\nSTATISTICS (Total images: {total})")
    print(f"{'Qwen Folder':<15} {'Match':<10} {'Differ':<10} {'Match %':<12} {'Precision':<12} {'Recall':<12}")
    
    # Print statistics for each available folder
    for folder in ['simple', 'reason']:
        if folder not in available_folders:
            continue
        
        s = stats[folder]
        match = s['match']
        differ = s['differ']
        tp = s['tp']
        fp = s['fp']
        fn = s['fn']
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        print(f"{folder.capitalize():<15} {match:<10} {differ:<10} {100*match/total:<11.1f}% {precision:<11.3f} {recall:<11.3f}")
    
    # Print detailed table header
    print(f"\nDETAILED COMPARISON (All {total} images)")
    
    # Build header based on available folders
    header_parts = [f"{'Idx':<5}", f"{'Manual':<12}"]
    for folder in available_folders:
        header_parts.append(f"{'Qwen('+folder+')':<15}")
        header_parts.append(f"{folder.capitalize()+' Status':<20}")
    print(" ".join(header_parts))
    
    # Print detailed rows
    for idx in sorted(all_indices):
        # Manual label
        if idx in incorrect:
            manual_label = "incorrect"
        elif idx in ambiguous:
            manual_label = "ambiguous"
        else:
            manual_label = "OK"
        
        manual_prob = manual_label in ['incorrect', 'ambiguous']
        
        row_parts = [f"{idx:<5}", f"{manual_label:<12}"]
        
        for folder in available_folders:
            pred = qwen_by_index.get(idx, {}).get(folder, 'N/A')
            pred_prob = pred in ['NO', 'UNCLEAR', 'AMBIGUOUS']
            
            # Status
            if manual_prob and pred_prob:
                status = "[OK] Match (both prob)"
            elif not manual_prob and not pred_prob:
                status = "[OK] Match (both OK)"
            elif manual_prob and not pred_prob:
                status = "[X] FN (missed)"
            else:  # not manual_prob and pred_prob
                status = "[X] FP (wrong)"
            
            row_parts.append(f"{pred:<15}")
            row_parts.append(f"{status:<20}")
        
        print(" ".join(row_parts))


In [ ]:
compare_image_level('above')

In [ ]:
compare_image_level('below')

In [ ]:
compare_image_level('behind')

In [ ]:
compare_image_level('to the left of')

In [ ]:
compare_image_level('to the right of')

## 5. New Directory Structure: Image-Level Viewer

Use this section for the new Qwen directory layout (`new_qwen_outputs/{model}/{strategy}`).
It shows selected images plus prompt, manual annotation, and Qwen annotation.

The manual annotations use the same tensor indices as the image tensors.